# What we want to learn today? 
Today we will have a first look at the Convokit toolkit for conversational analysis. The toolkit is designed to provide a unified framework to load and work with interactive conversational data. It already comes with a larger amount of existing conversational datasets, and it also provides a set of tools to manipulate, analyze, and model conversational data. 

In the session today we will look at two dataset that have annotations for two of the pragmatic phenomena we talked about in the first lecture: politeness strategies and speech acts. The website for Convokit is here: https://convokit.cornell.edu/

What we will cover today:
- representation of datasets: the corpus object in Convokit, how to load and extract information from it?
- manipulation of data: How to use the Transformation Pipeline (e.g., filtering, adding annotations, extracting linguistic features)?
- How to train a simple feature-based classifier to predict politeness strategies or speech acts in the conversations?

In [ ]:
from convokit import Corpus, download, PolitenessStrategies
import random

In [ ]:
# we are first going to download the two corpora we will be using today.
# 1) The Stanford Politeness Corpus (Danescu-Niculescu-Mizil et al., 2013), which contains ~10k posts from Stack Exchange annotated for politeness strategies.
pol_corpus = Corpus(filename=download("wikipedia-politeness-corpus"))
# 2) The Switchboard Corpus (Godfrey et al., 1992), which contains ~1k two-sided telephone conversations annotated for speech acts.
swbd_corpus = Corpus(filename=download("switchboard-corpus"))

If you want to learn more about the two corpora, you can check out the following links:
- Stanford Politeness Corpus: https://www.cs.cornell.edu/~cristian//pdfs/politeness_talk.pdf
- and the paper: https://aclanthology.org/P13-1025.pdf
- Switchboard Corpus: https://web.stanford.edu/~jurafsky/tr.pdf

In [ ]:
# Let's take a look at the summary statistics of the two corpora.
pol_corpus.print_summary_stats()
swbd_corpus.print_summary_stats()

Each corpus component has a consistent data format and we have three core components / objects:
- the utterance object: represents a single utterance in a conversation. It has primary data fields such as text, id, speaker, conversation_id (the conversation it belongs to), reply_to (the utterance it is replying to, if any), and metadata attributes such as annotations or additional features (e.g. dependency parse trees, sentiment scores, etc.).
- to access a random utterance from the corpus, we can use the .random_utterance() method.
- to access a specific utterance by its ID, we can use the .get_utterance(utterance_id) method.
- we can also iterate through all utterances in the corpus using the .iter_utterances() method.

**Task:** retrieve a random utterance from the politeness corpus and the switchboard corpus and look at its text, speaker, and metadata. What differences do you notice between the two corpora?


In [ ]:
utt_pol = pol_corpus.random_utterance()
utt_swbd = swbd_corpus.random_utterance()
print("Politeness Corpus Utterance:")
print(utt_pol.text)
print(utt_pol.speaker)
print(utt_pol.meta)
print("\nSwitchboard Corpus Utterance:")
print(utt_swbd.text)
print(utt_swbd.speaker)
print(utt_swbd.meta)

**Solution**: Difference: in the politeness dataset utterances are annotated with syntactic information. The utterances in the switchboard corpus are spoken language (thus, they have additional paralinguistic information).  

In [ ]:
# we can also get all utterances of a corpus as a pandas dataframe. This could be helpful for quick inspection or analysis on the dataset level.
utt_df_pol = pol_corpus.get_utterances_dataframe()
utt_df_swbd = swbd_corpus.get_utterances_dataframe()

In [ ]:
utt_df_pol.head()

In [ ]:
utt_df_swbd.head()

- the speaker object: represents a single speaker in a conversation. It has primary data fields such as id, and metadata attributes such as demographic information or other speaker-level annotations.

**Task**: retrieve a random speaker from the politeness corpus and the switchboard corpus and look at its id, and metadata. What differences do you notice between the two corpora?

In [ ]:
speaker_pol = pol_corpus.random_speaker()
speaker_swbd = swbd_corpus.random_speaker()
print("Politeness Corpus Speaker:")
print(speaker_pol.id)
print(speaker_pol.meta)
print("\nSwitchboard Corpus Speaker:")
print(speaker_swbd.id)
print(speaker_swbd.meta)

In [ ]:
# again we can retrieve all speakers as a dataframe.
speaker_df_pol = pol_corpus.get_speakers_dataframe()
speaker_df_swbd = swbd_corpus.get_speakers_dataframe()

In [ ]:
speaker_df_pol.head()

In [ ]:
speaker_df_swbd.head()

**Solution:** The speakers in the politeness corpus do not contain any information. In fact, it seems that there is only one speaker in the politeness corpus with always the same ID. 
On the other hand, the switchboard corpus contains additional metadata about different speakers, e.g. which dialect area they are located, which age and sex they have, which education. There are more than 1000 different speakers in the switchboard corpus.

- the third core object is the conversation object: it represents a single conversation / interaction. It has primary data fields such as id, owner (which corpus does it belong to?) and metadata attributes such as topic or other conversation-level annotations.
- to access a random conversation from the corpus, we can use the .random_conversation() method.
- to access a specific conversation by its ID, we can use the .get_conversation(conversation_id) method.
- we can also iterate through all conversations in the corpus using the .iter_conversations() method.
- we can also get all conversations of a corpus as a dataframe with the .get_conversations_dataframe() method.

**Task:** retrieve a random conversation from the politeness corpus and the switchboard corpus and look at its primary data fields and metadata and compare them.

In [ ]:
convo_pol = pol_corpus.random_conversation()
convo_swbd = swbd_corpus.random_conversation()
print("Politeness Corpus Conversation:")
print(convo_pol.id)
print(convo_pol.owner)
print(convo_pol.meta)
print("\nSwitchboard Corpus Conversation:")
print(convo_swbd.id)
print(convo_swbd.owner)
print(convo_swbd.meta)
print(convo_swbd.tree)

In [ ]:
convo_df_pol = pol_corpus.get_conversations_dataframe()
convo_df_swbd = swbd_corpus.get_conversations_dataframe()

In [ ]:
convo_df_pol.head()

In [ ]:
convo_df_swbd.head()

**Solution:** The politeness corpus again, does not contain any information on the conversations. In fact, the politeness coropus does not really represent a conversation, since the authors of the original work only extracted the original post and not the replies. Each post is thus annotated for politeness, but there is without conversational context.
The switchboard corpus on the other hand, contains rich metadata on the conversations, e.g. the topic of the conversation, the number of exchanges, speaker and recipient IDs. 

To summarize: every corpus in Convokit is made up of conversations, that contain utterances, which are produced by speakers. For each component we can access the corresponding connected components, such as speaker of an utterance, conversation of an utterance, utterances in a conversation, etc.

In the politeness corpus we have annotated politeness labels, stored in the utterance metadata. Take a look at the distribution of politeness labels in the corpus. The binarized politeness label ("Binary") indicates 1=”polite”, 0=”neutral”, -1 = “impolite”. The continuous politeness score ("PolitenessScore") is a continuous score ranging from -3 (most impolite) to 3 (most polite).

**Task:** check the distribution of the labels (e.g. plot a histogram or the value counts method from pandas) for both the binary and continuous politeness labels.

In [ ]:
utt_df_pol["meta.Binary"].value_counts()

In [ ]:
utt_df_pol["meta.Normalized Score"].hist(bins=30)


Next, we will look into how to use different transformers to manipulate and analyze the corpus data. 

Fist, we will use the built-in PolitenessStrategies transformer to extract, so called "politeness strategies" from the utterance text and add them as features to the utterance metadata. The transformer uses pattern-matching rules over syntactic parses to detect linguistic indicators of politeness. How can we use an transformer in Convokit?
1) First, we need to initialize the transformer object: pol_strategies = PolitenessStrategies()
2) Then, we can apply the transformer to the corpus using the .transform() method: pol_corpus = pol_strategies.transform(pol_corpus)
3) Finally, we can access the extracted features in the utterance metadata. The extracted politeness strategies are stored in the 'politeness_strategies' field of the utterance metadata as a dictionary, where keys are strategy names and values are lists of tuples containing the span indices and corresponding text tokens, if markers == True, otherwise only counts are stored.
4) we can also use the summarize() method of pol_strategies to get an overview of the extracted politeness strategies. (and use plot=True to visualize the distribution of strategies)

**Task:** follow the steps above to extract politeness strategies from the politeness corpus and display the first few utterances with their extracted strategies. Try to get and idea what different politeness strategies are annotated and how they look like in the utterances. Then use the summarize() method to visualize the distribution of strategies.

For a short / rough overview of the politeness strategies in this corpus: 

| Strategy                | Linguistic Pattern (simplified)      | Example                       |
|--------------------------|--------------------------------------|--------------------------------|
| **Apology**              | regex for “sorry”, “apologize”, etc. | “I’m sorry to ask, but…”       |
| **Gratitude**            | tokens like “thanks”, “thank you”    | “Thanks for your help!”        |
| **Deference**            | honorifics like “sir”, “ma’am”       | “Yes, sir.”                    |
| **Indirect Request**     | modal verbs “could”, “would” + verb  | “Could you check this?”        |
| **Please**               | token “please” anywhere              | “Please have a look.”          |
| **Hedges**               | “maybe”, “I think”, “sort of”        | “I think you might be right.”  |
| **First Person Plural**  | use of “we”, “our” for solidarity    | “We should try again.”         |
| **Compliment**           | “good job”, “nice work”, etc.        | “Great answer!”                |


In [ ]:
# initialization of a transformer object. This one is called PolitenessStrategies. All possible transfomers are listed here: 
# https://convokit.cornell.edu/documentation/transformers.html
pol_strategies = PolitenessStrategies()
# call the transform method to apply the transformer to the corpus
pol_corpus = pol_strategies.transform(pol_corpus, markers=True)

In [ ]:
# we can inspect the first few utterances with their extracted strategies. They are stored under the 'politeness_strategies' and 'politeness_markers' fields in the utterance metadata.
df_pol = pol_corpus.get_utterances_dataframe()
df_pol.head()

In [ ]:
# use the summarize() method of pol_strategies to get an overview of the extracted politeness strategies. For plot = True we can also visualize the distribution of strategies.
pol_strategies.summarize(pol_corpus, plot=True)

We can visualize the tokens that correspond to the politeness features / politeness strategies with the following code. This code shows a random utterance and features highlighted / color-coded. You can run this code a few times to see some examples from the corpus and the tokens highlighted.

In [ ]:
import re
from IPython.display import display, HTML

def highlight_politeness_markers(utterance):
    """
    Visualize politeness markers in an utterance.
    """
    text = utterance.text
    markers = utterance.meta.get('politeness_markers', {})

    # Collect all tokens to highlight
    highlight_spans = []
    for key, token_lists in markers.items():
        if not token_lists:
            continue
        strategy = key.replace('politeness_markers_==', '').replace('==', '')
        for token_group in token_lists:
            for token, sent_i, tok_i in token_group:
                highlight_spans.append((token, strategy))

    if not highlight_spans:
        print("No politeness markers detected for this utterance.")
        print(utterance.text)
        return

    colors = [
        "#FFD54F", "#AED581", "#81D4FA", "#CE93D8",
        "#FFAB91", "#C5E1A5", "#F48FB1", "#80CBC4",
        "#E6EE9C", "#B39DDB"
    ]
    color_map = {}
    color_index = 0

    def get_color(strategy):
        nonlocal color_index
        if strategy not in color_map:
            color_map[strategy] = colors[color_index % len(colors)]
            color_index += 1
        return color_map[strategy]

    highlighted_text = text
    for token, strategy in highlight_spans:
        color = get_color(strategy)
        pattern = r'\b' + re.escape(token) + r'\b'
        repl = f"<mark style='background-color:{color};' title='{strategy}'>{token}</mark>"
        highlighted_text = re.sub(pattern, repl, highlighted_text, flags=re.IGNORECASE)

    # Create color-coded legend
    legend_html = "<div style='margin-top:10px;'>"
    for strategy, color in color_map.items():
        legend_html += f"<span style='background-color:{color}; padding:2px 6px; border-radius:4px; margin-right:4px;'>{strategy}</span>"
    legend_html += "</div>"

    # Display the highlighted text + legend
    html = f"""
    <div style='font-family:sans-serif; line-height:1.6;'>
        <p>{highlighted_text}</p>
        {legend_html}
    </div>
    """
    display(HTML(html))

    #print("Detected strategies:", sorted(set([s for _, s in highlight_spans])))

u = list(pol_corpus.iter_utterances())[random.randint(0, len(pol_corpus.utterances) - 1)]
highlight_politeness_markers(u)

Now that we have extracted politeness strategies as features, we can use them to train a simple classifier to predict *whether an utterance is polite* or not. We will use the built-in Classifier module in Convokit for this purpose. We will use the extracted politeness strategies as features and the binary politeness label as the target variable.

In [ ]:
# import the Classifier. Important Note: For this wrapper to work properly, please make sure you have installed the previous version of Convokit (v.3.0.0), not the latest one. The latest one seem to have some issues with the Classifier wrapper.
from convokit import Classifier

The Classifier object:
- obj_type: specifies the type of object to classify (utterance, speaker, conversation)
- labeller: a function that takes an object and returns its label (in this case we want to use the .meta info of the utterance to get the binary politeness label)
- you can then use the functions .evaluate_with_cv() to get results on a 5-fold cross validation

**Task**: Use the Classifier to predict politeness based on the extracted politeness strategies. Use the binary politeness label as the target variable. Look at the 5-fold cross validation results.


An alternative solution is to not use the built-in wrapper of convokit but to use the sckit-learn classifier explicitly. Then you have more options to adapt the models to your use cases. For that you need to create the feature vectors from the meta data when iterating over the corpus. The label you can get with utterance.meta.get('Binary'). You can iterate over corpus utterances with .iter_utterances()

In [ ]:
# 1) first you need to remove the "neutral" utterances from the corpus, since we only want to classify polite vs. impolite utterances.
pol_binary_corpus = Corpus(utterances=[utt for utt in pol_corpus.iter_utterances() if utt.meta["Binary"] != 0])

In [ ]:
# 2) initialize the classifier. The features have to be specified in the pred_feats parameter. The labeller function retrieves the binary politeness label from the utterance metadata. Since we want to predict politeness vs. impoliteness, we have to define the positive class. The lamda function returns True for polite utterances if the meta data is label == 1 and False for impolite utterances (label == -1).
cv_classifier_politeness = Classifier(
    obj_type='utterance', pred_feats=['politeness_strategies'],
    labeller=lambda u: u.meta['Binary'] == 1)

# 3) evaluate the classifier with 5-fold cross-validation. The corpus that is evaluated needs to be specified.
cv_results = cv_classifier_politeness.evaluate_with_cv(pol_binary_corpus)
print("Cross-validation accuracy per fold:", cv_results)
print("Mean accuracy:", sum(cv_results) / len(cv_results))


Since we used a LogisticRegression Classifier we can directly inspect which features where most impactful. 

**Task:** Look at the 10 "most polite" and 10 "most impolite" indicators.

In [ ]:
import pandas as pd
feature_names = sorted(list(next(iter(pol_binary_corpus.iter_utterances())).meta['politeness_strategies'].keys()))
# we need to fit the classifier on the full data to get the coefficients
cv_classifier_politeness.fit_transform(
    corpus=pol_binary_corpus)
# extract the coefficients (weights) of the features
coefs = cv_classifier_politeness.clf.named_steps['logreg'].coef_[0]

# create a dataframe for better visualization
coef_df = pd.DataFrame({'Feature': feature_names, 'Weight': coefs})
# get all positive for polite, all negative for impolite
positive_features = coef_df[coef_df['Weight'] > 0].copy()
negative_features = coef_df[coef_df['Weight'] < 0].copy()
# sort by weight (absolute value)
positive_features['AbsWeight'] = positive_features['Weight'].abs()
positive_features = positive_features.sort_values('Weight', ascending=False)
negative_features['AbsWeight'] = negative_features['Weight'].abs()
negative_features = negative_features.sort_values('Weight')
# display top 10 polite and impolite features
print("Top 10 Polite Indicators:")
display(positive_features.head(10))
print("Top 10 Impolite Indicators:")
display(negative_features.head(10))

The next code block uses scikit-learn instead of the built in classifier of convo kit. We iterate over utterances and retrieve the politeness strategies as feature. 

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# Step 1: Extract features + labels from the ConvoKit corpus
X = []
y = []

for utt in pol_binary_corpus.iter_utterances():
    label = utt.meta.get('Binary')
    # extract politeness strategy features
    feats = utt.meta.get('politeness_strategies', {})
    # this sorts the features to ensure consistent ordering, then for each feature key we get the count (or 0 if not present). So we get a list of feature values, each value corresponding to the count of a specific politeness strategy in the utterance.
    X.append([feats.get(k, 0) for k in sorted(feats.keys())])
    # binarize the label: 1 for polite, 0 for impolite
    y.append(1 if label == 1 else 0)

# Convert to arrays
X = np.array(X)
y = np.array(y)

print("Feature matrix shape:", X.shape)
print("Labels distribution:", np.bincount(y))

In [ ]:
# Define a simple pipeline: scale + logistic regression
model_politeness = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))

# 5-fold cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model_politeness, X, y, cv=cv, scoring='accuracy')

# here we can compare the results with the built-in convokit classifier
print("Cross-validation accuracy per fold:", scores)
print("Mean accuracy:", scores.mean())


In [ ]:
# Fit once on the full data for inspection
model_politeness.fit(X, y)

# Extract coefficients
feature_names = sorted(list(next(iter(pol_binary_corpus.iter_utterances())).meta['politeness_strategies'].keys()))
coefs = model_politeness.named_steps['logisticregression'].coef_[0]

coef_df = pd.DataFrame({'Feature': feature_names, 'Weight': coefs})
# get all positive for polite, all negative for impolite
positive_features = coef_df[coef_df['Weight'] > 0].copy()
negative_features = coef_df[coef_df['Weight'] < 0].copy()
# sort by weight (absolute value)
positive_features['AbsWeight'] = positive_features['Weight'].abs()
positive_features = positive_features.sort_values('Weight', ascending=False)
negative_features['AbsWeight'] = negative_features['Weight'].abs()
negative_features = negative_features.sort_values('Weight')
# display top 10 polite and impolite features
print("Top 10 Polite Indicators:")
display(positive_features.head(10))
print("Top 10 Impolite Indicators:")
display(negative_features.head(10))


## Speech Acts

Next, let's have a look at speech acts. We will look at the switchboard corpus. We have to use a different version of the corpus for being able to use the speech acts. In this corpus, since it captures dialogues, the concept is called "dialogue acts".

**Task:** check how these are annotated in the dataset, the labels are stored in the "tags" field in the metadata. Inspect a few random utterances. 

In [ ]:
swbd_corpus_processed = Corpus(filename=download("switchboard-processed-corpus"))


In [ ]:
for u in random.sample(list(swbd_corpus_processed.iter_utterances()), 3):
    print(u.text)
    print("Tags:", u.meta.get('tags', []))
    print("-"*40)

For your reference, here is a table with the DMSL tags used in the Switchboard corpus, the full name of the tag and an example.

**Task:** In what way do they differ from the speech acts we have looked at in the lecture?

| DAMSL Tag | Dialogue / Speech Act Label          | Example Utterance                          |
|------------|--------------------------------------|--------------------------------------------|
| **sd**     | Statement-non-opinion               | “It’s raining outside.”                    |
| **sv**     | Statement-opinion                   | “I think that’s a bad idea.”               |
| **b**      | Acknowledge / Backchannel           | “Uh-huh.” / “Right.”                       |
| **aa**     | Agree / Accept                      | “Yes.” / “Exactly.”                        |
| **ba**     | Appreciative                        | “Thanks a lot!”                            |
| **qy**     | Yes-No Question                     | “Do you like pizza?”                       |
| **qw**     | Wh-Question                         | “What time is it?”                         |
| **qo**     | Open Question                       | “How do you feel about that?”              |
| **qr**     | Or-Question                         | “Is it red or blue?”                       |
| **qyd**    | Declarative Yes-No Question         | “You’re coming tonight?”                   |
| **ny**     | Yes Answer                          | “Yes.” / “I do.”                           |
| **nn**     | No Answer                           | “No.” / “I don’t.”                         |
| **fc**     | Conventional Closing                | “Goodbye.” / “See you later.”              |
| **fp**     | Conventional Opening                | “Hi!” / “Hello!”                           |
| **fe**     | Exclamation                         | “Wow!” / “Great!”                          |
| **ft**     | Thanking                            | “Thanks!” / “Thank you so much.”           |
| **fa**     | Apology                             | “Sorry about that.”                        |
| **fo**     | Offer / Invitation                  | “Would you like some coffee?”              |
| **oo**     | Other Answers                       | “Maybe.” / “Could be.”                     |
| **h**      | Hedge / Qualified Answer            | “I guess so.” / “Probably.”                |
| **na**     | Affirmative Non-Yes Answer          | “Yeah, sure.” / “Absolutely.”              |
| **ng**     | Negative Non-No Answer              | “Nope, not really.”                        |
| **arp**    | Repetition Request                  | “Pardon?” / “Could you repeat that?”       |
| **aa**     | Accept / Agree                      | “That’s true.”                             |
| **sd@**    | Statement continuation (multi-turn) | “And then we went to the park.”            |
| **t1**     | Self-Talk                           | “Let’s see…” / “Hmm…”                      |
| **t3**     | Segment Other                       | (Speaker changes topic)                    |
| **x**      | Non-verbal / Uninterpretable        | “[laughter]” / “[noise]”                   |



**Solution:** They seem to be very fine-grained. Several of them could maybe be mapped back to the coarse-grained speech act category from Searle, eg. apology, thanking could be examples of expressives. Some of them are very specific to the dialogue setting, e.g. conventional opening / closing, backchanneling, etc.

Next, we want to also train a simple classifier, to predict the dialogue acts based on the utterance text. For that we first need to create feature vectors from the utterance text (remember that before, we used linguistic indicators of politeness to predict whether an utterance is polite or not). Now, we will use a bag-of-words representation for predicting Dialogue Acts. Convokit provides a Bag-of-Words transformer that can be used to create vectors from the text of utterances. 

**Task:** initialize the BoWTransformer and add the vectors to the corpus

In [ ]:
from convokit import BoWTransformer
# fit the transformer on the switchboard corpus

vectorizer = BoWTransformer(obj_type='utterance')
swbd_corpus_processed = vectorizer.fit_transform(swbd_corpus_processed)


You can now see that the utterances_dataframe has a new column, called "vectors". With utt.get_vector("bow_vector") we can retrieve the vector representation for an utterance. We can now use any scikit-learn classifier and try to predict dialogue acts based on the bag-of-words representation. Note: training the classifier might take a while on you local machine. If it takes too long, only use a subset of the dataset. 

**Task:** Use a Classifier to learn to predict Dialogue Acts. You can either use the built-in Classifier wrapper from Convokit or use scikit-learn directly. Which dialogue acts are easiest to predict? Which ones are hardest? Inspect some of the misclassifications. Can you find a pattern? If you use the convokit classifier wrapper class, use the VectorClassifier and specify clf=LogisticRegression(solver='lbfgs', max_iter=1000), which is a logistic regression classifier that uses a one-vs-rest strategy for multi-class classification.

In [ ]:
# you can now see that the utterances_dataframe has a new column, called "vectors". with utt.get_vector("bow_ve
swbd_corpus_utterances = swbd_corpus_processed.get_utterances_dataframe()
swbd_corpus_utterances.head()

In [ ]:
# use the VectorClassifier from convokit. We now have a multi-class classification problem, since there are more than 2 dialogue act labels. So we need to specify a multi-class classifier. Here we use LogisticRegression with default settings, which uses a one-vs-rest strategy for multi-class classification.
from convokit import VectorClassifier
vector_classifier = VectorClassifier(
    obj_type='utterance',
    vector_name="bow_vector",
    labeller=lambda u: u.meta['tags'][0] if 'tags' in u.meta and len(u.meta['tags']) > 0 else None,
    clf=LogisticRegression(solver='lbfgs', max_iter=1000)
)
# evaluate with train-test split, we need to use the selector to only select utterances with tags, otherwise we would get an error.
accuracy, confusion_matrix = vector_classifier.evaluate_with_train_test_split(swbd_corpus_processed, selector=lambda u: 'tags' in u.meta and len(u.meta['tags']) > 0)
# now get the corpus portion used for testing by checking the predictions



In [ ]:
print("Accuracy on Switchboard Dialogue Acts:", accuracy)

**Solution for using scikit-learn directly:**
The next code block shows how to use scikit-learn directly to train a classifier for dialogue act prediction.

In [ ]:
# 1) we first need to extract the feature vectors and labels from the corpus
X = []
y = []
for utt in swbd_corpus_processed.iter_utterances():
    if 'tags' in utt.meta and len(utt.meta['tags']) > 0:
        label = utt.meta['tags'][0]  # use the first tag as label
        vector = utt.get_vector("bow_vector").toarray().flatten()  # get the BoW vector
        X.append(vector)
        y.append(label)

# Convert to arrays
X = np.array(X)
y = np.array(y)

print("Feature matrix shape:", X.shape)

In [ ]:
print("Labels distribution:", pd.Series(y).value_counts())

In [ ]:
# Define a simple pipeline for using the BoW vectors + logistic regression
model_dialogue_acts = make_pipeline(StandardScaler(), LogisticRegression(solver='lbfgs', max_iter=1000))

# define a train and test split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
# fit the model (this might take a while, you can also use a subset of the data for faster training)
model_dialogue_acts.fit(X_train, y_train)
# predict on the test set
y_pred = model_dialogue_acts.predict(X_test)
# evaluate the accuracy
from sklearn.metrics import accuracy_score, classification_report
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy on Switchboard Dialogue Acts:", accuracy)
# print the classification report
print(classification_report(y_test, y_pred))

You have now a first idea of how to load data with convokit and extract features. Pick one of the following tasks

(a) Use a classifier trained to predict politeness based on the politeness features to predict it on a new dataset. You can either use the other portion of the corpus (stack-exchange-politeness-corpus) 
or 

(b) the switchboard corpus. This is spoken language, so a very different domain / genre. 
Inspect the predictions on a subset of 20 utterances. Do they make sense? What is the classifier missing?

Final question: in general, this is a rather simple classifier. How do you think the classifier could be improved? Is there any additional context, for example that could be taken into consideration?

# Solution a)

In [ ]:
# solution 1: predicting politeness on the stack-exchange-politeness-corpus
new_pol_corpus = Corpus(filename=download("stack-exchange-politeness-corpus"))

In [ ]:
# extract politeness strategies from the new corpus
new_pol_corpus = pol_strategies.transform(new_pol_corpus, markers=True)



In [ ]:
# create feature vectors. 
X = []

for utt in new_pol_corpus.iter_utterances():
    # extract politeness strategy features
    feats = utt.meta.get('politeness_strategies', {})
    # this sorts the features to ensure consistent ordering, then for each feature key we get the count (or 0 if not present). So we get a list of feature values, each value corresponding to the count of a specific politeness strategy in the utterance.
    X.append([feats.get(k, 0) for k in sorted(feats.keys())])

# Convert to arrays
X = np.array(X)
print("Feature matrix shape:", X.shape)

In [ ]:
# predict politeness on the new corpus
predictions_new_politeness = model_politeness.predict(X)

In [ ]:
# inspect a few predictions on a subset of 20 utterances
for utt, pred in zip(random.sample(list(new_pol_corpus.iter_utterances()), 20), predictions_new_politeness):
    print(utt.text)
    # display the politeness features
    strategies = utt.meta.get('politeness_strategies', {})
    # filter only those strategies that are present (count > 0)
    present_strategies = {k: v for k, v in strategies.items() if v > 0}
    print("Extracted Politeness Strategies:", present_strategies)
    print("Predicted Politeness:", "Polite" if pred == 1 else "Impolite")
    print("-"*40)

# Solution b)

In [ ]:
# to add parses we need some space model locally, so we first download the small english model from spacy
import spacy
spacy.cli.download("en_core_web_sm")

In [ ]:
# 1) the politeness transfomer expects text with syntactic parses. We need to first add syntactic parses to the switchboard corpus. Therefore we will use the TextParser transformer from convokit. This transformer uses spacy to add syntactic parses to the utterance text. It requires spacy to be installed and a spacy model to be downloaded. This might take a while.
from convokit.text_processing import TextParser

parser = TextParser()
swbd_corpus_with_parses = parser.transform(swbd_corpus_processed)


In [ ]:
# 2) extract politeness strategies from the switchboard corpus with parses
swbd_corpus_with_politeness = pol_strategies.transform(swbd_corpus_with_parses, markers=True)

In [ ]:
# 3) create feature vectors.
X = []
for utt in swbd_corpus_with_politeness.iter_utterances():
    # extract politeness strategy features
    feats = utt.meta.get('politeness_strategies', {})
    X.append([feats.get(k, 0) for k in sorted(feats.keys())])
# Convert to arrays
X = np.array(X)
print("Feature matrix shape:", X.shape)

In [ ]:
predictions_politeness_swbd = model_politeness.predict(X)


In [ ]:
# inspect a few predictions on a subset of 20 utterances
for utt, pred in zip(random.sample(list(swbd_corpus_with_politeness.iter_utterances()), 20), predictions_politeness_swbd):
    print(utt.text)
    # display the politeness features
    strategies = utt.meta.get('politeness_strategies', {})
    # filter only those strategies that are present (count > 0)
    present_strategies = {k: v for k, v in strategies.items() if v > 0}
    print("Extracted Politeness Strategies:", present_strategies)
    print("Predicted Politeness:", "Polite" if pred == 1 else "Impolite")
    print("-"*40)